# Overview

This notebook is designed for embeddings extraction from the images.
This notebook is especially designed to be started on MacBook devices.


## 1. Importing

Import libs and set up the settings.

In [14]:
import os

import torch
import numpy as np
import pandas as pd

from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, AutoModel

In [15]:
# Original model requires auth and request for accessing the model.
# Another author made a fork with free access but this fork was removed from HF.
# Fortunately, fork was cached on my device so I use this local snapshot. Close enough, huh.
MODEL_NAME = "/Users/locked15/.cache/huggingface/hub/models--camenduru--dinov3-vitl16-pretrain-lvd1689m/snapshots/3c276edd87d6f6e569ff0c4400e086807d0f3881"

BASE_FOLDER = "/Users/locked15/Repos.localized/Python/Study/UUST/Course 3/Semester 2/Practice/sludge-utilities/apps/ml/ml-training_lab/"

INTERIM_DATASET_FOLDER = "data/meta/interim/"
PROCESSED_DATASET_FOLDER = "data/meta/processed/gb/"
DATASET_IMAGES_FOLDER = "data/images/"

TARGET_LBA_COLUMN = "lba_image_path"
TARGET_SLUDGE_COLUMN = "sludge_image_path"

TARGET_INPUT_FILE = "metadata_augmented.parquet"
TARGET_OUTPUT_FILE = "metadata_dinov3_embeddings"


In [16]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}.")

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()

Using device: mps.


Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

DINOv3ViTModel(
  (embeddings): DINOv3ViTEmbeddings(
    (patch_embeddings): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
  )
  (rope_embeddings): DINOv3ViTRopePositionEmbedding()
  (model): DINOv3ViTEncoder(
    (layer): ModuleList(
      (0-23): 24 x DINOv3ViTLayer(
        (norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
        (attention): DINOv3ViTAttention(
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (o_proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (layer_scale1): DINOv3ViTLayerScale()
        (drop_path): Identity()
        (norm2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): DINOv3ViTMLP(
          (up_proj): Linear(in_features=1024, out_features=4096, bias=True)
          (down_proj

## 2. Dataset Preparation

We prepare the dataset.

In [17]:
class SludgeDataset(Dataset):
    def __init__(self, df, base_dir, images_dir, image_col):
        self.processor = processor
        if base_dir:
            self.paths = (os.path.join(base_dir, images_dir) + df[image_col]).tolist()
        else:
            self.paths = (images_dir + df[image_col]).tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = Image.open(self.paths[idx]).convert("RGB")
        inputs = self.processor(images = image, return_tensors = "pt")
        return {k: v.squeeze(0) for k, v in inputs.items()}

## 3. Inference Settings

In [18]:
def extract_embeddings(df, base_dir, images_dir, image_col, batch_size = 8):
    dataset = SludgeDataset(df, base_dir, images_dir, image_col)
    loader = DataLoader(dataset, batch_size = batch_size, shuffle = False,
                        num_workers = 0, pin_memory = False)
    
    embeddings = []
    with torch.no_grad():
        for batch in tqdm(loader):
            pixel_values = batch['pixel_values'].to(device)
            outputs = model(pixel_values)
            emb = outputs.last_hidden_state[:, 0]
            embeddings.append(emb.cpu().numpy())
    return np.vstack(embeddings)

## 4. Start-Up

In [19]:
print("Initializing model and processor...")

print("Loading dataset...")
df = pd.read_parquet(os.path.join(BASE_FOLDER, INTERIM_DATASET_FOLDER, TARGET_INPUT_FILE))
print("Dataset loaded.")

print("Extracting embeddings (lba)...")
lba_emb = extract_embeddings(df, BASE_FOLDER, DATASET_IMAGES_FOLDER, TARGET_LBA_COLUMN, batch_size = 8)
print("Extracting embeddings (sludge)...")
sludge_emb = extract_embeddings(df, BASE_FOLDER, DATASET_IMAGES_FOLDER, TARGET_SLUDGE_COLUMN, batch_size = 8)
print("Embeddings extracted.")

print("Inserting embeddings into DataFrame...")
df['sludge_dinov3_emb'] = list(sludge_emb)
df['lba_dinov3_emb'] = list(lba_emb)

print("Saving dataset with embeddings...")
df.to_csv(os.path.join(BASE_FOLDER, PROCESSED_DATASET_FOLDER, f"{TARGET_OUTPUT_FILE}.csv"), index = False)
df.to_parquet(os.path.join(BASE_FOLDER, PROCESSED_DATASET_FOLDER, f"{TARGET_OUTPUT_FILE}.parquet"), index = False)
print("Dataset with embeddings saved.")

Initializing model and processor...
Loading dataset...
Dataset loaded.
Extracting embeddings (lba)...


 10%|▉         | 28/287 [00:08<01:15,  3.41it/s]


KeyboardInterrupt: 